In [1]:
!nvidia-smi


'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
%%writefile vector_add.cu
#include <iostream>
#include <cuda_runtime.h>
#include <chrono>
using namespace std;

__global__ void add(int *a, int *b, int *c, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;

    if (i < n) {
        c[i] = a[i] + b[i];
    }
}

int main() {

    int n = 10000000;   // large input: 1 crore elements
    int size = n * sizeof(int);

    int *a = new int[n];
    int *b = new int[n];
    int *c_seq = new int[n];
    int *c_gpu = new int[n];

    for (int i = 0; i < n; i++) {
        a[i] = i;
        b[i] = i * 2;
    }

    // Sequential
    auto s1 = chrono::high_resolution_clock::now();

    for (int i = 0; i < n; i++) {
        c_seq[i] = a[i] + b[i];
    }

    auto s2 = chrono::high_resolution_clock::now();

    // GPU
    int *d_a, *d_b, *d_c;

    cudaMalloc(&d_a, size);
    cudaMalloc(&d_b, size);
    cudaMalloc(&d_c, size);

    # auto g1 = chrono::high_resolution_clock::now();

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    int blockSize = 256;
    int gridSize = n / blockSize + 1;

    auto g1 = chrono::high_resolution_clock::now();
    add<<<gridSize, blockSize>>>(d_a, d_b, d_c, n);
    auto g2 = chrono::high_resolution_clock::now();
    
    cudaDeviceSynchronize();

    cudaMemcpy(c_gpu, d_c, size, cudaMemcpyDeviceToHost);

    # auto g2 = chrono::high_resolution_clock::now();

    cout << "First 10 GPU results:\n";
    for (int i = 0; i < 10; i++) {
        cout << c_gpu[i] << " ";
    }

    cout << "\n\nSequential Time: "
         << chrono::duration<double>(s2 - s1).count()
         << " sec";

    cout << "\nGPU Time: "
         << chrono::duration<double>(g2 - g1).count()
         << " sec\n";

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    delete[] a;
    delete[] b;
    delete[] c_seq;
    delete[] c_gpu;

    return 0;
}

Writing vector_add.cu


In [3]:
!nvcc vector_add.cu -o vector_add


'nvcc' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
!./vector_add

'.' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
%%writefile matrix_mul.cu
#include <iostream>
#include <cuda_runtime.h>
#include <chrono>

using namespace std;
using namespace chrono;

#define N 500

__global__ void matMul(int *A, int *B, int *C)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N)
    {
        int sum = 0;

        for (int k = 0; k < N; k++)
        {
            sum += A[row * N + k] *
                   B[k * N + col];
        }

        C[row * N + col] = sum;
    }
}

int main()
{
    int size = N * N * sizeof(int);

    int *A = new int[N * N];
    int *B = new int[N * N];
    int *C_seq = new int[N * N];
    int *C_gpu = new int[N * N];

    // Fill matrices
    for (int i = 0; i < N * N; i++)
    {
        A[i] = 1;
        B[i] = 1;
    }

    // ---------------- CPU ----------------

    auto start_seq = high_resolution_clock::now();

    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            int sum = 0;

            for (int k = 0; k < N; k++)
            {
                sum += A[i * N + k] *
                       B[k * N + j];
            }

            C_seq[i * N + j] = sum;
        }
    }

    auto end_seq = high_resolution_clock::now();

    // ---------------- GPU ----------------

    int *d_A, *d_B, *d_C;

    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    cudaMemcpy(d_A, A, size,
               cudaMemcpyHostToDevice);

    cudaMemcpy(d_B, B, size,
               cudaMemcpyHostToDevice);

    dim3 threads(16,16);
    dim3 blocks((N + 15)/16,
                (N + 15)/16);

    auto start_gpu = high_resolution_clock::now();

    matMul<<<blocks, threads>>>(d_A, d_B, d_C);

    cudaDeviceSynchronize();

    auto end_gpu = high_resolution_clock::now();

    cudaMemcpy(C_gpu, d_C, size,
               cudaMemcpyDeviceToHost);

    // ---------------- OUTPUT ----------------

    cout << "First 10 GPU Results:\n";

    for (int i = 0; i < 10; i++)
    {
        cout << C_gpu[i] << " ";
    }

    cout << "\n\nSequential Time: "
         << duration<double>(end_seq - start_seq).count()
         << " sec";

    cout << "\nGPU Time: "
         << duration<double>(end_gpu - start_gpu).count()
         << " sec\n";

    // ---------------- CLEANUP ----------------

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    delete[] A;
    delete[] B;
    delete[] C_seq;
    delete[] C_gpu;

    return 0;
}

Writing matrix_mul.cu


In [6]:
!nvcc matrix_mul.cu -o matrix_mul

'nvcc' is not recognized as an internal or external command,
operable program or batch file.


In [7]:
!./matrix_mul

'.' is not recognized as an internal or external command,
operable program or batch file.
